<a href="https://colab.research.google.com/github/BrajanNieto/CompuVision/blob/main/CV_Labo2_ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Computer Vision - Segmentación de imágenes y detección de bordes**

---



*   Este notebook se enfoca en aplicar técnicas fundamentales de visión por computadora para el procesamiento de imágenes. Incluye tres tareas principales: segmentación basada en color de frutas, detección de bordes en imágenes médicas de rayos X, y segmentación de señales de tráfico combinando umbrales de color y métodos de detección de bordes. El objetivo es practicar transformaciones RGB/HSV, filtrado y operadores clásicos como Sobel y Canny.

**Autores:**  

Nieto Espinoza, Brajan E.  
[brajan.nieto@utec.edu.pe](mailto:brajan.nieto@utec.edu.pe)

Guedes del Pozo,  Rodrigo J.  
[rodrigo.guedes.d@utec.edu.pe](mailto:rodrigo.guedes.d@utec.edu.pe)

<img src="https://pregrado.utec.edu.pe/sites/default/files/logo-utec-h_0_0.svg" width="190" alt="Logo UTEC" loading="lazy" typeof="foaf:Image">      

---

In [ ]:
# === Libraries===
import cv2
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen

### 1. Cargar la imagen y visualizarla en el formato RGB correcto.

In [ ]:
# ============================================================
# CELDA 1 (EJECUTAR PRIMERO)
# Objetivo:
# 1) Cargar CIFAR-10 con cifar10.load_data() (valores por defecto).
# 2) Quedarnos SOLO con 3 clases (configurables abajo).
# 3) Separar el conjunto resultante en 80% train y 20% test (estratificado).
#
# Instrucciones:
# - Ejecuta esta celda una vez. Cambia SELECTED_CLASSES si quieres otras 3.
# - Por defecto se usan: 0=airplane, 1=automobile, 8=ship.
# ============================================================

import numpy as np
import random
import cv2
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Cargar CIFAR-10 (por defecto: (50000 train, 10000 test) de 10 clases)
(X_train_full, y_train_full), (X_test_full, y_test_full) = cifar10.load_data()

# Elegimos 3 clases (puedes cambiarlas)
# 0='airplane',1='automobile',2='bird',3='cat',4='deer',
# 5='dog',6='frog',7='horse',8='ship',9='truck'
SELECTED_CLASSES = [0, 1, 8]  # airplane, automobile, ship

CLASS_NAMES = {
    0:'airplane', 1:'automobile', 2:'bird', 3:'cat', 4:'deer',
    5:'dog', 6:'frog', 7:'horse', 8:'ship', 9:'truck'
}

# Usaremos SOLO el split de entrenamiento original de Keras
X_full = X_train_full
y_full = y_train_full.flatten()

# Filtrar a las 3 clases elegidas
mask = np.isin(y_full, SELECTED_CLASSES)
X = X_full[mask]
y = y_full[mask]

# (Opcional) Si quieres limitar muestras por clase para acelerar, define N_PER_CLASS (ej. 2000). None = usar todas.
N_PER_CLASS = None
if N_PER_CLASS is not None:
    X_sel, y_sel = [], []
    for c in SELECTED_CLASSES:
        idx_c = np.where(y == c)[0]
        np.random.shuffle(idx_c)
        idx_c = idx_c[:N_PER_CLASS]
        X_sel.append(X[idx_c])
        y_sel.append(y[idx_c])
    X = np.concatenate(X_sel, axis=0)
    y = np.concatenate(y_sel, axis=0)

# Separación 80/20 estratificada
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Info
def dist_clases(vec):
    uniques, counts = np.unique(vec, return_counts=True)
    return {CLASS_NAMES[int(k)]: int(v) for k, v in zip(uniques, counts)}

print("Clases seleccionadas:", [f"{c}={CLASS_NAMES[c]}" for c in SELECTED_CLASSES])
print("Dimensiones X_train:", X_train.shape, "X_test:", X_test.shape)
print("Distribución train:", dist_clases(y_train))
print("Distribución test :", dist_clases(y_test))


### 2. Convertir la imagen a un espacio de color HSV.

In [ ]:
# ============================================================
# CELDA 2
# Objetivo:
# 1) Definir una función que extraiga 8 características por imagen:
#    R, G, B, H, S, V, bordes Sobel horizontales y verticales.
# 2) Visualizar estos 8 "canales" para UNA imagen aleatoria.
# 3) Construir matrices de características para train y test.
#
# Instrucciones:
# - Ejecuta esta celda después de la Celda 1.
# - Cambia SHOW_INDEX si quieres ver otra imagen.
# ============================================================

import numpy as np
import cv2
import matplotlib.pyplot as plt

def extract_8_features(img_rgb):
    """
    img_rgb: (32,32,3) uint8 en espacio RGB.
    Retorna un vector 1D con la concatenación de:
    [R, G, B, H, S, V, Sobel_H (dy=1), Sobel_V (dx=1)]  (todos aplanados).
    """
    # Canales RGB
    R = img_rgb[:, :, 0].astype(np.float32)
    G = img_rgb[:, :, 1].astype(np.float32)
    B = img_rgb[:, :, 2].astype(np.float32)

    # HSV (OpenCV espera RGB->HSV con cv2.COLOR_RGB2HSV)
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H = hsv[:, :, 0].astype(np.float32)   # [0..179] en OpenCV
    S = hsv[:, :, 1].astype(np.float32)   # [0..255]
    V = hsv[:, :, 2].astype(np.float32)   # [0..255]

    # Bordes Sobel en escala de grises
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    sobel_h = cv2.Sobel(gray, ddepth=cv2.CV_64F, dx=0, dy=1, ksize=3)  # cambios horizontales
    sobel_v = cv2.Sobel(gray, ddepth=cv2.CV_64F, dx=1, dy=0, ksize=3)  # cambios verticales
    sobel_h = cv2.convertScaleAbs(sobel_h).astype(np.float32)
    sobel_v = cv2.convertScaleAbs(sobel_v).astype(np.float32)

    # Concatenar y aplanar
    feats = np.stack([R, G, B, H, S, V, sobel_h, sobel_v], axis=0)  # (8, 32, 32)
    return feats.reshape(-1)  # (8*32*32,)

def batch_extract_features(X):
    return np.vstack([extract_8_features(img) for img in X])

# Visualización de los 8 canales para 1 imagen aleatoria del set de entrenamiento
SHOW_INDEX = np.random.randint(0, len(X_train))
img = X_train[SHOW_INDEX]

R = img[:, :, 0]
G = img[:, :, 1]
B = img[:, :, 2]
hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
Hc, Sc, Vc = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
sobel_h = cv2.convertScaleAbs(cv2.Sobel(gray, ddepth=cv2.CV_64F, dx=0, dy=1, ksize=3))
sobel_v = cv2.convertScaleAbs(cv2.Sobel(gray, ddepth=cv2.CV_64F, dx=1, dy=0, ksize=3))

titles = ["R", "G", "B", "H", "S", "V", "Sobel H (dy=1)", "Sobel V (dx=1)"]
planes = [R, G, B, Hc, Sc, Vc, sobel_h, sobel_v]

plt.figure(figsize=(12, 6))
for i, (t, p) in enumerate(zip(titles, planes)):
    ax = plt.subplot(2, 4, i+1)
    ax.imshow(p, cmap='gray')
    ax.set_title(t)
    ax.axis('off')
plt.suptitle(f"8 canales de una imagen '{CLASS_NAMES[int(y_train[SHOW_INDEX])]}'")
plt.tight_layout()
plt.show()

# Construir features para train y test
print("Extrayendo características (esto puede tardar un poco)...")
X_train_feat = batch_extract_features(X_train)
X_test_feat  = batch_extract_features(X_test)
print("X_train_feat:", X_train_feat.shape, "X_test_feat:", X_test_feat.shape)


### 3. Visualizar las bandas R, G, B, H, S, y V.

In [ ]:
# ============================================================
# CELDA 3
# Objetivo:
# 3.1 Entrenar REGRESIÓN LOGÍSTICA con Grid Search (accuracy y f1_macro):
#     - Imprimir top resultados de CV.
#     - Evaluar el MEJOR por accuracy y el MEJOR por f1_macro en TEST:
#       * classification_report
#       * matriz de confusión
#     - Indicar cuál consideramos "mejor" (por accuracy, configurable).
# 3.2 Con el mejor, mostrar 15 imágenes aleatorias del TEST con su predicción,
#     indicando si acierta o no en el título.
#
# Instrucciones:
# - Ejecuta después de la Celda 2.
# - Puedes cambiar METRICA_DECISION a 'accuracy' o 'f1_macro'.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.base import clone

METRICA_DECISION = 'accuracy'  # 'accuracy' o 'f1_macro'

# Pipeline y grilla
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=1000, multi_class='ovr', n_jobs=None))
])

param_grid_lr = {
    'logreg__C': [0.01, 0.1, 1, 10, 100],
    'logreg__solver': ['liblinear', 'lbfgs', 'saga'],
    'logreg__penalty': ['l2'],
    'logreg__class_weight': [None, 'balanced'],
}

scoring = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro'}

gs_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    scoring=scoring,
    refit='accuracy',   # reentrena el mejor según accuracy
    cv=3,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

gs_lr.fit(X_train_feat, y_train)

# Mostrar top-5 por accuracy y por f1_macro (CV)
cv = gs_lr.cv_results_
def top_k(cv, key, k=5):
    idx = np.argsort(cv[f"mean_test_{key}"])[::-1][:k]
    filas = []
    for i in idx:
        filas.append({
            'rank': int(i),
            'mean_'+key: float(cv[f"mean_test_{key}"][i]),
            'params': cv['params'][i]
        })
    return filas

print("\n=== TOP-5 (CV) por accuracy ===")
for row in top_k(cv, 'accuracy', 5):
    print(row)

print("\n=== TOP-5 (CV) por f1_macro ===")
for row in top_k(cv, 'f1_macro', 5):
    print(row)

# Mejor por accuracy (ya refiteado)
best_acc_est = gs_lr.best_estimator_
y_pred_acc = best_acc_est.predict(X_test_feat)

print("\n=== LOGISTIC REGRESSION — Mejor por ACCURACY (en TEST) ===")
print(classification_report(y_test, y_pred_acc, target_names=[CLASS_NAMES[c] for c in SELECTED_CLASSES]))

cm_acc = confusion_matrix(y_test, y_pred_acc, labels=SELECTED_CLASSES)
plt.figure(figsize=(4,4))
plt.imshow(cm_acc, interpolation='nearest')
plt.title('Matriz de confusión (mejor por accuracy)')
plt.xticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES], rotation=45)
plt.yticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES])
plt.colorbar()
plt.tight_layout()
plt.show()

# Mejor por f1_macro (reentrenamos esa configuración)
idx_f1 = int(np.argmax(cv['mean_test_f1_macro']))
best_params_f1 = cv['params'][idx_f1]
best_f1_est = clone(pipe_lr).set_params(**{f"logreg__{k.split('__')[-1]}": v for k, v in best_params_f1.items()})
best_f1_est.fit(X_train_feat, y_train)
y_pred_f1 = best_f1_est.predict(X_test_feat)

print("\n=== LOGISTIC REGRESSION — Mejor por F1_MACRO (en TEST) ===")
print(classification_report(y_test, y_pred_f1, target_names=[CLASS_NAMES[c] for c in SELECTED_CLASSES]))

cm_f1 = confusion_matrix(y_test, y_pred_f1, labels=SELECTED_CLASSES)
plt.figure(figsize=(4,4))
plt.imshow(cm_f1, interpolation='nearest')
plt.title('Matriz de confusión (mejor por f1_macro)')
plt.xticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES], rotation=45)
plt.yticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES])
plt.colorbar()
plt.tight_layout()
plt.show()

# Elegimos el "mejor" según METRICA_DECISION
if METRICA_DECISION == 'f1_macro':
    best_lr = best_f1_est
    y_pred_best = y_pred_f1
    tag_best = "F1_MACRO"
else:
    best_lr = best_acc_est
    y_pred_best = y_pred_acc
    tag_best = "ACCURACY"

print(f"\n>>> Para seguir (3.2) usamos el modelo de LR con MEJOR {tag_best} en TEST.")

# 3.2 — Mostrar 15 imágenes del TEST con predicción y acierto/error
n_vis = 15
idxs = np.random.choice(len(X_test), size=n_vis, replace=False)

plt.figure(figsize=(12, 7))
for i, idx in enumerate(idxs, 1):
    img = X_test[idx]
    true_c = int(y_test[idx])
    pred_c = int(y_pred_best[idx])
    correcto = (true_c == pred_c)
    titulo = f"Pred: {CLASS_NAMES[pred_c]} {'✓' if correcto else '✗'}\nTrue: {CLASS_NAMES[true_c]}"
    ax = plt.subplot(3, 5, i)
    ax.imshow(img)
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')
plt.suptitle("Regresión Logística — 15 muestras de TEST")
plt.tight_layout()
plt.show()


###4. Experimentar con diferentes umbrales en las bandas para poder segmentar de la mejor forma una de las frutas mencionadas.

In [ ]:
# ============================================================
# CELDA 4
# Objetivo:
# 4.1 Entrenar KNN con Grid Search (accuracy y f1_macro):
#     - Imprimir top resultados de CV.
#     - Evaluar el MEJOR por accuracy y el MEJOR por f1_macro en TEST:
#       * classification_report
#       * matriz de confusión
#     - Indicar cuál consideramos "mejor" (por accuracy, configurable).
# 4.2 Con el mejor, mostrar 15 imágenes aleatorias del TEST con su predicción,
#     indicando si acierta o no en el título.
#
# Instrucciones:
# - Ejecuta después de la Celda 3 (o tras la 2 si quieres comparar directo).
# - Puedes cambiar METRICA_DECISION_KNN a 'accuracy' o 'f1_macro'.
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.base import clone

METRICA_DECISION_KNN = 'accuracy'  # 'accuracy' o 'f1_macro'

pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2],  # Manhattan (1) o Euclídea (2)
    'knn__leaf_size': [15, 30, 45],
}

scoring = {'accuracy': 'accuracy', 'f1_macro': 'f1_macro'}

gs_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid_knn,
    scoring=scoring,
    refit='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=1,
    return_train_score=False
)

gs_knn.fit(X_train_feat, y_train)

# Mostrar top-5 por accuracy y por f1_macro (CV)
cvk = gs_knn.cv_results_
def top_k(cv, key, k=5):
    idx = np.argsort(cv[f"mean_test_{key}"])[::-1][:k]
    filas = []
    for i in idx:
        filas.append({
            'rank': int(i),
            'mean_'+key: float(cv[f"mean_test_{key}"][i]),
            'params': cv['params'][i]
        })
    return filas

print("\n=== TOP-5 (CV) por accuracy (KNN) ===")
for row in top_k(cvk, 'accuracy', 5):
    print(row)

print("\n=== TOP-5 (CV) por f1_macro (KNN) ===")
for row in top_k(cvk, 'f1_macro', 5):
    print(row)

# Mejor por accuracy (ya refiteado)
best_acc_knn = gs_knn.best_estimator_
y_pred_acc_knn = best_acc_knn.predict(X_test_feat)

print("\n=== KNN — Mejor por ACCURACY (en TEST) ===")
print(classification_report(y_test, y_pred_acc_knn, target_names=[CLASS_NAMES[c] for c in SELECTED_CLASSES]))

cm_acc_knn = confusion_matrix(y_test, y_pred_acc_knn, labels=SELECTED_CLASSES)
plt.figure(figsize=(4,4))
plt.imshow(cm_acc_knn, interpolation='nearest')
plt.title('Matriz de confusión (KNN mejor por accuracy)')
plt.xticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES], rotation=45)
plt.yticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES])
plt.colorbar()
plt.tight_layout()
plt.show()

# Mejor por f1_macro (reentrenamos esa configuración)
idx_f1_knn = int(np.argmax(cvk['mean_test_f1_macro']))
best_params_f1_knn = cvk['params'][idx_f1_knn]
best_f1_knn = clone(pipe_knn).set_params(**{f"knn__{k.split('__')[-1]}": v for k, v in best_params_f1_knn.items()})
best_f1_knn.fit(X_train_feat, y_train)
y_pred_f1_knn = best_f1_knn.predict(X_test_feat)

print("\n=== KNN — Mejor por F1_MACRO (en TEST) ===")
print(classification_report(y_test, y_pred_f1_knn, target_names=[CLASS_NAMES[c] for c in SELECTED_CLASSES]))

cm_f1_knn = confusion_matrix(y_test, y_pred_f1_knn, labels=SELECTED_CLASSES)
plt.figure(figsize=(4,4))
plt.imshow(cm_f1_knn, interpolation='nearest')
plt.title('Matriz de confusión (KNN mejor por f1_macro)')
plt.xticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES], rotation=45)
plt.yticks(range(len(SELECTED_CLASSES)), [CLASS_NAMES[c] for c in SELECTED_CLASSES])
plt.colorbar()
plt.tight_layout()
plt.show()

# Elegimos el "mejor" según METRICA_DECISION_KNN
if METRICA_DECISION_KNN == 'f1_macro':
    best_knn = best_f1_knn
    y_pred_best_knn = y_pred_f1_knn
    tag_best_knn = "F1_MACRO"
else:
    best_knn = best_acc_knn
    y_pred_best_knn = y_pred_acc_knn
    tag_best_knn = "ACCURACY"

print(f"\n>>> Para (4.2) usamos el modelo KNN con MEJOR {tag_best_knn} en TEST.")

# 4.2 — Mostrar 15 imágenes del TEST con predicción y acierto/error
n_vis = 15
idxs = np.random.choice(len(X_test), size=n_vis, replace=False)

plt.figure(figsize=(12, 7))
for i, idx in enumerate(idxs, 1):
    img = X_test[idx]
    true_c = int(y_test[idx])
    pred_c = int(y_pred_best_knn[idx])
    correcto = (true_c == pred_c)
    titulo = f"Pred: {CLASS_NAMES[pred_c]} {'✓' if correcto else '✗'}\nTrue: {CLASS_NAMES[true_c]}"
    ax = plt.subplot(3, 5, i)
    ax.imshow(img)
    ax.set_title(titulo, fontsize=9)
    ax.axis('off')
plt.suptitle("KNN — 15 muestras de TEST")
plt.tight_layout()
plt.show()


###5. Visualizar tanto la imagen original en formato RGB como la máscara final de segmentación de la fruta